# Build an auditable LLC decision workflow

This notebook is for power-electronics engineers who want to change a specification and see how analytical rules become machine-readable decisions. It rebuilds every quick-demo output from configuration, seed, equations and thresholds.

**Boundary:** this is preliminary electrical screening at one fixed operating point. It is not hardware approval or higher-fidelity validation.

## 1. Change only these parameters

Start with the defaults. Then change one value, choose **Restart and Run All**, and compare the generated report.

In [ ]:
VIN_V = 400.0          # V
VOUT_V = 36.0         # V
POUT_W = 230.0        # W
CANDIDATE_COUNT = 20  # quick demo
RUN_LABEL = None      # None creates a timestamped output

## 2. Validate the engineering contract and rebuild

The next cell loads the published configuration, applies your four inputs, validates units and ranges, generates new candidates, runs FHA and evaluates the survivors with the linear-equivalent model. LTspice is not required.

In [ ]:
from datetime import datetime
from pathlib import Path
import copy, csv, json, sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llc_tool.config import load_and_validate_config, validate_config
from llc_tool.workflow import run_workflow

config = load_and_validate_config(ROOT / 'configs' / 'quick_demo.json')
config['input_spec']['vin_v'] = VIN_V
config['input_spec']['vout_v'] = VOUT_V
config['input_spec']['pout_w'] = POUT_W
config['candidate_count'] = CANDIDATE_COUNT
config = validate_config(config)

label = RUN_LABEL or f"notebook-demo-{datetime.now():%Y%m%d-%H%M%S}"
OUTPUT = ROOT.parent / 'LLC_Notebook_Runs' / label
result = run_workflow(config, OUTPUT)
result

## 3. Read the gates before reading the score

A valid FHA calculation can reject a candidate. A completed time-stepped evaluation can pass or reject its gate. A failed method must remain `failed` with `gate_result=not_applicable`. Engineering approval remains pending in every automatic record.

In [ ]:
from IPython.display import HTML, Markdown, SVG, display

summary = json.loads((OUTPUT / 'summary.json').read_text(encoding='utf-8'))
verification = json.loads((OUTPUT / 'verification.json').read_text(encoding='utf-8'))
display(Markdown(
    f"**Output integrity:** {verification['status']}  \
"
    f"**Generated:** {summary['candidate_count']}  \
"
    f"**FHA:** {summary['fha_counts']}  \
"
    f"**Automatic gate:** {summary['gate_counts']}"
))
display(SVG(filename=str(OUTPUT / 'plots' / 'workflow.svg')))
display(SVG(filename=str(OUTPUT / 'plots' / 'gate_counts.svg')))
if summary['gate_counts']['pass'] == 0:
    display(Markdown('**0 candidates passed the configured gate.** The current design space does not contain a surviving candidate. Review turns ratio, frequency range or tank-variable bounds.'))

## 4. Explore the generated design space and model outputs

The plots show the sampled `Ln`–`Q` space, completed model outputs, FHA gain/phase explanations and—when a pass exists—the retained time-domain waveform. The commutation and power-ratio fields remain model proxies, not hardware measurements.

In [ ]:
for plot_name in ('design_space.svg', 'model_outputs.svg', 'fha_pass_case.svg', 'fha_reject_case.svg', 'pass_case_waveform.svg'):
    plot_path = OUTPUT / 'plots' / plot_name
    if plot_path.is_file():
        display(SVG(filename=str(plot_path)))

## 5. Compare the available pass, reject and failed cases

The forced failure is an explicitly synthetic software test. It is not a converter result and is excluded from campaign counts and ML rows.

In [ ]:
cases = json.loads((OUTPUT / 'pedagogical_cases.json').read_text(encoding='utf-8'))
rows = []
for case in cases:
    record = case['record']
    contract = record['decision_contract']
    fha = record.get('fha_screen', {})
    time_domain = record.get('time_domain_evaluation', {})
    rows.append(
        '<tr>'
        f"<td><code>{record['candidate_id']}</code></td>"
        f"<td>{case['case_type']}</td>"
        f"<td><code>{fha.get('execution_status', 'not_applicable')}</code></td>"
        f"<td><code>{fha.get('gate_result', 'not_applicable')}</code></td>"
        f"<td><code>{time_domain.get('execution_status', 'not_applicable')}</code></td>"
        f"<td><code>{contract['gate_result']}</code></td>"
        f"<td><code>{contract['engineering_approval']}</code></td>"
        '</tr>'
    )
display(HTML(
    '<table><thead><tr><th>Record</th><th>Case</th><th>FHA execution</th><th>FHA gate</th><th>Time-domain execution</th><th>Final automated disposition</th><th>Engineering</th></tr></thead>'
    + '<tbody>' + ''.join(rows) + '</tbody></table>'
))

## 6. Inspect the ML-oriented export

The export separates hardware-only design identity, operating point and method run. Targets are blank whenever the time-domain method was not run or failed. Hardware designs are assigned as groups to train, validation or test before any model is fitted.

In [ ]:
with (OUTPUT / 'ml_dataset_v1.csv').open('r', newline='', encoding='utf-8') as stream:
    preview = list(csv.DictReader(stream))[:5]
columns = ['candidate_id', 'design_id', 'split', 'execution_status', 'gate_result', 'target_available', 'resonant_current_rms_a']
head = ''.join(f'<th>{name}</th>' for name in columns)
body = ''.join('<tr>' + ''.join(f'<td>{row[name]}</td>' for name in columns) + '</tr>' for row in preview)
display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))
display(Markdown(f"Open the full human report: `{OUTPUT / 'summary.html'}`"))

## Exercise

Change one input in the first code cell and rerun the notebook. Before looking at the results, predict which gate count will change and why. Then inspect the reason codes instead of judging only the final count.

The next Field Note should add controlled operating-point diversity and a completed higher-fidelity comparison before using these rows as a serious surrogate-training dataset.